<a href="https://colab.research.google.com/github/drod75/nyc_taxi_analysis/blob/main/notebooks/data_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Engineering

The point of this notebook is to use our data we collected to create new data. In particular we will take our taxi trip data and turn it into trip metric data, showcasing metrics for each unique trip within the data, and unique statistics per each day of the summer.

## Setup

In [1]:
# importing libraries, pandas for data manipulation, requests for data gathering, colab for environment variables
import pandas as pd
from google.colab import userdata
from google.colab import drive
from pprint import pprint as print

In [2]:
# mounting drive for future use
drive.mount('drive')

Drive already mounted at drive; to attempt to forcibly remount, call drive.mount("drive", force_remount=True).


In [3]:
# importing libraries
taxi_df = pd.read_csv("/content/drive/MyDrive/taxi_summer_data.csv")
zone_df = pd.read_json("/content/drive/MyDrive/taxi_zones_data.json")

## Individual Trips Statistics

This branch of our summer data will condense every unique trip and gather there aggregrated statistics, such as fare data, trip length data, if it was in the borough, etc.

In [4]:
taxi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11707066 entries, 0 to 11707065
Data columns (total 18 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   tpep_pickup_datetime   object 
 1   tpep_dropoff_datetime  object 
 2   passenger_count        float64
 3   trip_distance          float64
 4   ratecodeid             float64
 5   pulocationid           int64  
 6   dolocationid           int64  
 7   payment_type           int64  
 8   fare_amount            float64
 9   extra                  float64
 10  mta_tax                float64
 11  tip_amount             float64
 12  tolls_amount           float64
 13  improvement_surcharge  float64
 14  total_amount           float64
 15  congestion_surcharge   float64
 16  airport_fee            float64
 17  cbd_congestion_fee     float64
dtypes: float64(13), int64(3), object(2)
memory usage: 1.6+ GB


In [5]:
trip_df = taxi_df.groupby(['pulocationid', 'dolocationid']).agg(
    trip_count=('tpep_pickup_datetime', 'size'),
    avg_passenger_count=('passenger_count', 'mean'),
    median_passenger_count=('passenger_count', 'median'),
    min_passenger_count=('passenger_count', 'min'),
    max_passenger_count=('passenger_count', 'max'),
    avg_trip_distance=('trip_distance', 'mean'),
    median_trip_distance=('trip_distance', 'median'),
    min_trip_distance=('trip_distance', 'min'),
    max_trip_distance=('trip_distance', 'max'),
    avg_total_amount=('total_amount', 'mean'),
    median_total_amount=('total_amount', 'median'),
    min_total_amount=('total_amount', 'min'),
    max_total_amount=('total_amount', 'max'),
)
trip_df.head()

trip_count  avg_passenger_count  \
pulocationid dolocationid                                    
1            1                   1553             1.828811   
             23                     1             1.000000   
             25                     1             3.000000   
             40                     1             1.000000   
             48                     4             1.750000   

                           median_passenger_count  min_passenger_count  \
pulocationid dolocationid                                                
1            1                                1.0                  0.0   
             23                               1.0                  1.0   
             25                               3.0                  3.0   
             40                               1.0                  1.0   
             48                               1.5                  1.0   

                           max_passenger_count  avg_trip_distance  \
pulocationid dolocationid                                           
1            1                             9.0           0.141397   
             23                            1.0          12.440000   
             25                            3.0          12.940000   
             40                            1.0          17.230000   
             48                            3.0          16.127500   

                           median_trip_distance  min_trip_distance  \
pulocationid dolocationid                                            
1            1                             0.00               0.00   
             23                           12.44              12.44   
             25                           12.94              12.94   
             40                           17.23              17.23   
             48                           16.63              13.55   

                           max_trip_distance  avg_total_amount  \
pulocationid dolocationid                                        
1            1                         25.10         93.346452   
             23                        12.44         78.770000   
             25                        12.94        199.250000   
             40                        17.23        145.540000   
             48                        17.70         86.252500   

                           median_total_amount  min_total_amount  \
pulocationid dolocationid                                          
1            1                          113.26           -261.00   
             23                          78.77             78.77   
             25                         199.25            199.25   
             40                         145.54            145.54   
             48                          95.21             46.00   

                           max_total_amount  
pulocationid dolocationid                    
1            1                       313.20  
             23                       78.77  
             25                      199.25  
             40                      145.54  
             48                      108.59

In [6]:
trip_df.reset_index(inplace=True)
trip_df.head()

,pulocationid,dolocationid,trip_count,avg_passenger_count,median_passenger_count,min_passenger_count,max_passenger_count,avg_trip_distance,median_trip_distance,min_trip_distance,max_trip_distance,avg_total_amount,median_total_amount,min_total_amount,max_total_amount
0,1,1,1553,1.828811,1.0,0.0,9.0,0.141397,0.00,0.00,25.10,93.346452,113.26,-261.00,313.20
1,1,23,1,1.000000,1.0,1.0,1.0,12.440000,12.44,12.44,12.44,78.770000,78.77,78.77,78.77
2,1,25,1,3.000000,3.0,3.0,3.0,12.940000,12.94,12.94,12.94,199.250000,199.25,199.25,199.25
3,1,40,1,1.000000,1.0,1.0,1.0,17.230000,17.23,17.23,17.23,145.540000,145.54,145.54,145.54
4,1,48,4,1.750000,1.5,1.0,3.0,16.127500,16.63,13.55,17.70,86.252500,95.21,46.00,108.59


In [7]:
trip_df.sort_values(by=['pulocationid','dolocationid'], ascending=True, inplace=True)
trip_df.head(10)

,pulocationid,dolocationid,trip_count,avg_passenger_count,median_passenger_count,min_passenger_count,max_passenger_count,avg_trip_distance,median_trip_distance,min_trip_distance,max_trip_distance,avg_total_amount,median_total_amount,min_total_amount,max_total_amount
0,1,1,1553,1.828811,1.0,0.0,9.0,0.141397,0.00,0.00,25.10,93.346452,113.26,-261.00,313.20
1,1,23,1,1.000000,1.0,1.0,1.0,12.440000,12.44,12.44,12.44,78.770000,78.77,78.77,78.77
2,1,25,1,3.000000,3.0,3.0,3.0,12.940000,12.94,12.94,12.94,199.250000,199.25,199.25,199.25
3,1,40,1,1.000000,1.0,1.0,1.0,17.230000,17.23,17.23,17.23,145.540000,145.54,145.54,145.54
4,1,48,4,1.750000,1.5,1.0,3.0,16.127500,16.63,13.55,17.70,86.252500,95.21,46.00,108.59
5,1,50,2,1.000000,1.0,1.0,1.0,16.960000,16.96,16.65,17.27,148.110000,148.11,106.96,189.26
6,1,68,1,1.000000,1.0,1.0,1.0,14.100000,14.10,14.10,14.10,78.310000,78.31,78.31,78.31
7,1,79,1,2.000000,2.0,2.0,2.0,15.410000,15.41,15.41,15.41,96.760000,96.76,96.76,96.76
8,1,107,1,1.000000,1.0,1.0,1.0,15.760000,15.76,15.76,15.76,99.760000,99.76,99.76,99.76
9,1,113,1,1.000000,1.0,1.0,1.0,14.010000,14.01,14.01,14.01,122.060000,122.06,122.06,122.06


In [8]:
trip_df = pd.merge(trip_df, zone_df, left_on='pulocationid', right_on='locationid')
trip_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44902 entries, 0 to 44901
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   pulocationid            44902 non-null  int64  
 1   dolocationid            44902 non-null  int64  
 2   trip_count              44902 non-null  int64  
 3   avg_passenger_count     24355 non-null  float64
 4   median_passenger_count  24355 non-null  float64
 5   min_passenger_count     24355 non-null  float64
 6   max_passenger_count     24355 non-null  float64
 7   avg_trip_distance       44902 non-null  float64
 8   median_trip_distance    44902 non-null  float64
 9   min_trip_distance       44902 non-null  float64
 10  max_trip_distance       44902 non-null  float64
 11  avg_total_amount        44902 non-null  float64
 12  median_total_amount     44902 non-null  float64
 13  min_total_amount        44902 non-null  float64
 14  max_total_amount        44902 non-null

In [9]:
trip_df.drop(columns=['the_geom', 'shape_leng', 'shape_area', 'locationid'], inplace=True)
trip_df.rename(columns={"zone": "puzone", "borough": "puborough"}, inplace=True)
trip_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44902 entries, 0 to 44901
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   pulocationid            44902 non-null  int64  
 1   dolocationid            44902 non-null  int64  
 2   trip_count              44902 non-null  int64  
 3   avg_passenger_count     24355 non-null  float64
 4   median_passenger_count  24355 non-null  float64
 5   min_passenger_count     24355 non-null  float64
 6   max_passenger_count     24355 non-null  float64
 7   avg_trip_distance       44902 non-null  float64
 8   median_trip_distance    44902 non-null  float64
 9   min_trip_distance       44902 non-null  float64
 10  max_trip_distance       44902 non-null  float64
 11  avg_total_amount        44902 non-null  float64
 12  median_total_amount     44902 non-null  float64
 13  min_total_amount        44902 non-null  float64
 14  max_total_amount        44902 non-null

In [10]:
trip_df = pd.merge(trip_df, zone_df, left_on='dolocationid', right_on='locationid')
trip_df.drop(columns=['the_geom', 'shape_leng', 'shape_area', 'locationid'], inplace=True)
trip_df.rename(columns={"zone": "dozone", "borough": "doborough"}, inplace=True)
trip_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44604 entries, 0 to 44603
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   pulocationid            44604 non-null  int64  
 1   dolocationid            44604 non-null  int64  
 2   trip_count              44604 non-null  int64  
 3   avg_passenger_count     24056 non-null  float64
 4   median_passenger_count  24056 non-null  float64
 5   min_passenger_count     24056 non-null  float64
 6   max_passenger_count     24056 non-null  float64
 7   avg_trip_distance       44604 non-null  float64
 8   median_trip_distance    44604 non-null  float64
 9   min_trip_distance       44604 non-null  float64
 10  max_trip_distance       44604 non-null  float64
 11  avg_total_amount        44604 non-null  float64
 12  median_total_amount     44604 non-null  float64
 13  min_total_amount        44604 non-null  float64
 14  max_total_amount        44604 non-null

In [11]:
trip_df.head()

,pulocationid,dolocationid,trip_count,avg_passenger_count,median_passenger_count,min_passenger_count,max_passenger_count,avg_trip_distance,median_trip_distance,min_trip_distance,max_trip_distance,avg_total_amount,median_total_amount,min_total_amount,max_total_amount,puzone,puborough,dozone,doborough
0,1,1,1553,1.828811,1.0,0.0,9.0,0.141397,0.00,0.00,25.10,93.346452,113.26,-261.00,313.20,Newark Airport,EWR,Newark Airport,EWR
1,1,23,1,1.000000,1.0,1.0,1.0,12.440000,12.44,12.44,12.44,78.770000,78.77,78.77,78.77,Newark Airport,EWR,Bloomfield/Emerson Hill,Staten Island
2,1,25,1,3.000000,3.0,3.0,3.0,12.940000,12.94,12.94,12.94,199.250000,199.25,199.25,199.25,Newark Airport,EWR,Boerum Hill,Brooklyn
3,1,40,1,1.000000,1.0,1.0,1.0,17.230000,17.23,17.23,17.23,145.540000,145.54,145.54,145.54,Newark Airport,EWR,Carroll Gardens,Brooklyn
4,1,48,4,1.750000,1.5,1.0,3.0,16.127500,16.63,13.55,17.70,86.252500,95.21,46.00,108.59,Newark Airport,EWR,Clinton East,Manhattan


In [12]:
trip_df['interborough'] = (trip_df['puborough'] == trip_df['doborough'])
trip_df.head()

,pulocationid,dolocationid,trip_count,avg_passenger_count,median_passenger_count,min_passenger_count,max_passenger_count,avg_trip_distance,median_trip_distance,min_trip_distance,max_trip_distance,avg_total_amount,median_total_amount,min_total_amount,max_total_amount,puzone,puborough,dozone,doborough,interborough
0,1,1,1553,1.828811,1.0,0.0,9.0,0.141397,0.00,0.00,25.10,93.346452,113.26,-261.00,313.20,Newark Airport,EWR,Newark Airport,EWR,True
1,1,23,1,1.000000,1.0,1.0,1.0,12.440000,12.44,12.44,12.44,78.770000,78.77,78.77,78.77,Newark Airport,EWR,Bloomfield/Emerson Hill,Staten Island,False
2,1,25,1,3.000000,3.0,3.0,3.0,12.940000,12.94,12.94,12.94,199.250000,199.25,199.25,199.25,Newark Airport,EWR,Boerum Hill,Brooklyn,False
3,1,40,1,1.000000,1.0,1.0,1.0,17.230000,17.23,17.23,17.23,145.540000,145.54,145.54,145.54,Newark Airport,EWR,Carroll Gardens,Brooklyn,False
4,1,48,4,1.750000,1.5,1.0,3.0,16.127500,16.63,13.55,17.70,86.252500,95.21,46.00,108.59,Newark Airport,EWR,Clinton East,Manhattan,False


In [13]:
trip_df['interborough'].value_counts()

,count
interborough,
False,30931
True,13673


## Individual Time Statistics

This branch of our summer data contains the metric for all rides within a specific day, each row of the data being a day in the period of our data.

In [18]:
taxi_df['tpep_pickup_datetime'] = pd.to_datetime(taxi_df['tpep_pickup_datetime'])
taxi_df['tpep_dropoff_datetime'] = pd.to_datetime(taxi_df['tpep_dropoff_datetime'])

day_df = taxi_df.groupby(taxi_df['tpep_pickup_datetime'].dt.floor('d')).agg(
    total_trips=('tpep_pickup_datetime', 'size'),
    avg_passenger_count=('passenger_count', 'mean'),
    median_passenger_count=('passenger_count', 'median'),
    min_passenger_count=('passenger_count', 'min'),
    max_passenger_count=('passenger_count', 'max'),
    sum_passenger_count=('passenger_count', 'sum'),
    avg_trip_distance=('trip_distance', 'mean'),
    median_trip_distance=('trip_distance', 'median'),
    min_trip_distance=('trip_distance', 'min'),
    max_trip_distance=('trip_distance', 'max'),
    avg_total_amount=('total_amount', 'mean'),
    median_total_amount=('total_amount', 'median'),
    min_total_amount=('total_amount', 'min'),
    max_total_amount=('total_amount', 'max')
)
day_df.reset_index(inplace=True)
day_df.rename(columns={"tpep_pickup_datetime": "date"}, inplace=True)
day_df.head()

,date,total_trips,avg_passenger_count,median_passenger_count,min_passenger_count,max_passenger_count,sum_passenger_count,avg_trip_distance,median_trip_distance,min_trip_distance,max_trip_distance,avg_total_amount,median_total_amount,min_total_amount,max_total_amount
0,2025-06-21,156241,1.427008,1.0,0.0,6.0,140683.0,8.946772,2.20,0.0,218906.23,25.566632,21.10,-929.29,2502.94
1,2025-06-22,134833,1.380049,1.0,0.0,9.0,117064.0,8.793792,2.40,0.0,162065.95,27.675926,21.15,-904.25,904.25
2,2025-06-23,139800,1.261720,1.0,0.0,8.0,124451.0,7.337263,1.87,0.0,233336.85,30.504188,23.31,-804.25,1633.25
3,2025-06-24,155798,1.235461,1.0,0.0,6.0,136789.0,8.277978,1.70,0.0,255868.53,29.785436,23.58,-601.00,607.90
4,2025-06-25,156447,1.239272,1.0,0.0,6.0,142064.0,7.033034,1.70,0.0,153861.93,30.138291,23.94,-504.25,504.25


## Export

In [19]:
trip_df.to_csv('/content/drive/MyDrive/trip_data.csv', index=False)
day_df.to_csv('/content/drive/MyDrive/trip_day_data.csv', index=False)